In [ ]:
import os
import glob
import cv2
import pandas as pd
from deepface import DeepFace

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Importar las bibliotecas necesarias
import cv2
import os
import glob
import pandas as pd
from deepface import DeepFace

# Diccionario de emociones.
emotions_dict = {
    'English': {'Angry': 'Angry', 'Disgust': 'Disgust', 'Fear': 'Fear', 'Sad': 'Sad', 'Neutral': 'Neutral', 'Surprise': 'Surprise', 'Happy': 'Happy'},
    'Spanish': {'Angry': 'Enojado', 'Disgust': 'Asco', 'Fear': 'Miedo', 'Sad': 'Triste', 'Neutral': 'Neutral', 'Surprise': 'Sorpresa', 'Happy': 'Feliz'},
    'French': {'Angry': 'En colère', 'Disgust': 'Dégoût', 'Fear': 'Peur', 'Sad': 'Triste', 'Neutral': 'Neutre', 'Surprise': 'Surprise', 'Happy': 'Heureux'},
    'Portuguese': {'Angry': 'Bravo', 'Disgust': 'Desgosto', 'Fear': 'Medo', 'Sad': 'Triste', 'Neutral': 'Neutro', 'Surprise': 'Surpresa', 'Happy': 'Feliz'},
    'German': {'Angry': 'Wutend', 'Disgust': 'Ekel', 'Fear': 'Angst', 'Sad': 'Traurig', 'Neutral': 'Neutral', 'Surprise': 'Uberraschung', 'Happy': 'Glucklich'},
    'Italian': {'Angry': 'Arrabbiato', 'Disgust': 'Disgusto', 'Fear': 'Paura', 'Sad': 'Triste', 'Neutral': 'Neutrale', 'Surprise': 'Sorpresa', 'Happy': 'Felice'},
    'Dutch': {'Angry': 'Boos', 'Disgust': 'Afschuw', 'Fear': 'Vrees', 'Sad': 'Treurig', 'Neutral': 'Neutraal', 'Surprise': 'Verrassing', 'Happy': 'Gelukkig'},
    'Turkish': {'Angry': 'Kizgin', 'Disgust': 'Tiksinme', 'Fear': 'Korku', 'Sad': 'Uzgun', 'Neutral': 'Notr', 'Surprise': 'Sasirtici', 'Happy': 'Mutlu'},
    'Swedish': {'Angry': 'Arg', 'Disgust': 'Äcklad', 'Fear': 'Rädd', 'Sad': 'Ledsen', 'Neutral': 'Neutral', 'Surprise': 'Överraskad', 'Happy': 'Glad'},
    'Indonesian': {'Angry': 'Marah', 'Disgust': 'Jijik', 'Fear': 'Takut', 'Sad': 'Sedih', 'Neutral': 'Netral', 'Surprise': 'Kejutan', 'Happy': 'Bahagia'},
}




emotions_dict_lower = {lang: {emotion.lower(): desc for emotion, desc in emotions.items()} 
                       for lang, emotions in emotions_dict.items()}


# ANALIZADOR DE EMOCIONES V3.
def analizar_emociones_videos(ruta_carpeta_videos, sample_rate=1):
    ruta_temp_data = os.path.join(ruta_carpeta_videos, "temp_data")
    if not os.path.exists(ruta_temp_data):
        os.makedirs(ruta_temp_data)

    lista_archivos_videos = glob.glob(os.path.join(ruta_carpeta_videos, '*.mp4'))
    emociones_por_tiempo = {}

    for video_path in lista_archivos_videos:
        video_name = os.path.basename(video_path)
        emociones_por_tiempo[video_name] = {}
        vidcap = cv2.VideoCapture(video_path)
        fps = vidcap.get(cv2.CAP_PROP_FPS)
        sec = 0
        success = True

        while success:
            sec += sample_rate
            vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
            success, image = vidcap.read()
            if success:
                image_path = os.path.join(ruta_temp_data, f"image_{sec}.jpg")
                cv2.imwrite(image_path, image)
                try:
                    analysis = DeepFace.analyze(img_path=image_path, actions=['emotion'], enforce_detection=False)
                    emociones = analysis[0]['emotion'] if isinstance(analysis, list) else analysis['emotion']
                    emociones_por_tiempo[video_name][sec] = emociones
                except Exception as e:
                    print(f"Error en el segundo {sec}: {e}")

        vidcap.release()

    # Limpieza del directorio temporal
    for item in os.listdir(ruta_temp_data):
        os.remove(os.path.join(ruta_temp_data, item))
       
    #print(f"Tipo de emociones_por_tiempo: {type(emociones_por_tiempo)}")
    #print(f"Contenido de emociones_por_tiempo: {emociones_por_tiempo}")
    return emociones_por_tiempo    
    







# Función para insertar emociones en las imágenes
def insertar_emocion(image, emociones, umbral, idioma):
    # Encuentra la emoción dominante y su valor
    dominant_emotion, dominant_value = max(emociones.items(), key=lambda item: item[1])
    if dominant_value >= umbral:
        # Traduce la emoción al idioma deseado
        emocion_traducida = emotions_dict_lower[idioma].get(dominant_emotion.lower(), dominant_emotion.capitalize())
        # Configura los parámetros para mostrar el texto
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 1.0
        thickness = 2
        color = (255, 0, 0)  # Color rojo para mayor visibilidad
        text = f"{emocion_traducida}: {dominant_value:.2f}%"
        # Calcula la posición del texto
        text_size = cv2.getTextSize(text, font, scale, thickness)[0]
        text_x = (image.shape[1] - text_size[0]) // 2
        text_y = text_size[1] + 20  # Coloca el texto un poco abajo del top del marco
        # Inserta el texto en el frame
        cv2.putText(image, text, (text_x, text_y), font, scale, color, thickness)
        print(f"Subtítulo insertado: {text}")  # Mensaje de depuración
        return True  # Subtítulo insertado
    return False  # No se insertó subtítulo










# Función para generar video con subtítulos
def generar_video_con_subtitulos(ruta_carpeta_videos, emociones_por_tiempo, umbral, idioma, sample_rate):
    # Procesa cada video en el diccionario de emociones
    for video_name, emociones_dict in emociones_por_tiempo.items():
        ruta_video = os.path.join(ruta_carpeta_videos, video_name)
        vidcap = cv2.VideoCapture(ruta_video)
        fps = vidcap.get(cv2.CAP_PROP_FPS)
        frame_width = int(vidcap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(vidcap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fourcc = cv2.VideoWriter_fourcc(*'MP4V')
        out = cv2.VideoWriter(ruta_video.replace('.mp4', '_con_emociones.mp4'), fourcc, fps, (frame_width, frame_height))

        frame_number = 0
        while True:
            success, image = vidcap.read()
            if not success:
                break
            sec = frame_number / fps
            # Redondea 'sec' al múltiplo más cercano de 'sample_rate' para coincidir con las claves del diccionario
            sec_rounded = round(sec / sample_rate) * sample_rate
            if sec_rounded in emociones_dict:
                emociones = emociones_dict[sec_rounded]
                # Verifica si la emoción dominante supera el umbral y si es así, inserta el subtítulo
                if emociones['dominant_emotion'] and emociones[emociones['dominant_emotion']] >= umbral:
                    insertar_emocion(image, emociones, umbral, idioma)
            out.write(image)
            frame_number += 1

        vidcap.release()
        out.release()
        print(f"Video procesado: {video_name}")  # Mensaje de depuración
        
        
# Ahora, la ejecución del script:
umbral = 20  # Un umbral más bajo para probar
sample_rate = 0.25  # Asegúrate de que sample_rate está definido
resultado = analizar_emociones_videos("C:/Users/franc/Desktop/DeepVideoFER/Gestion", sample_rate=sample_rate)

if isinstance(resultado, dict):
    generar_video_con_subtitulos("C:/Users/franc/Desktop/DeepVideoFER/Gestion", resultado, umbral, 'German', sample_rate)
else:
    print("El resultado no es un diccionario.")


In [ ]:
resultado

In [ ]:
# Ejecución del análisis y generación de video con subtítulos
resultado = analizar_emociones_videos("C:/Users/franc/Desktop/DeepVideoFER/Gestion")

generar_video_con_subtitulos(resultado, "C:/Users/franc/Desktop/DeepVideoFER/Gestion", 80, 'German')
print("Ready!")

# EmoGraph


**Arreglar el frame rate. dejar el que viene por defecto del video. FPS.**

In [ ]:
def graficar_emociones_por_video(df, umbral=80, frame_rate=0.033333):
    import matplotlib.pyplot as plt
    """
    Grafica las emociones de un DataFrame a lo largo del tiempo (en segundos), 
    resaltando solo aquellos valores que superen un umbral específico.
    Se crea un gráfico por video.

    Parámetros:
    df (DataFrame): DataFrame que contiene las emociones y el nombre del video.
    umbral (float): Valor mínimo que una emoción debe tener para ser graficada.
    frame_rate (float): Duración en segundos de cada frame.
    """
    # Lista de columnas de emociones
    columnas_emociones = ['angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']

    # Por cada video en el DataFrame
    for video_name, group in df.groupby('Video'):
        # Establecer el estilo de plot
        plt.style.use('ggplot')

        # Establecer tamaño del gráfico
        plt.figure(figsize=(15, 7))

        # Calcular eje x en segundos
        x_seconds = [i * frame_rate for i in range(len(group))]

        # Graficar cada emoción
        for emocion in columnas_emociones:
            # Filtramos los datos por el umbral
            y = group[emocion].where(group[emocion] > umbral)
            plt.plot(x_seconds, y, label=emocion)

        # Establecer marcas y cuadrícula en el eje x
        max_time = max(x_seconds)
        ticks_interval = 5  # Intervalo de las marcas en segundos
        x_ticks = np.arange(0, max_time + ticks_interval, ticks_interval)
        plt.xticks(x_ticks)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)


        # Establecer etiquetas y título
        plt.title(f'Progreso de Emociones en Video: {video_name}')
        plt.xlabel('Tiempo (segundos)')
        plt.ylabel('Valor de Emoción (%)')
        plt.ylim(50, 100)  # Asegurarse que el eje y empiece en 50
        plt.legend(loc='upper right')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)

        # Mostrar el gráfico
        plt.tight_layout()
        plt.show()

def graficar_sub_emociones_por_video(df, umbral=80, frame_rate=0.033333):
    import numpy as np
    import matplotlib.pyplot as plt
    """
    Grafica las emociones de un DataFrame a lo largo del tiempo (en segundos), 
    resaltando solo aquellos valores que superen un umbral específico.
    Se crea un subgráfico por emoción y un gráfico general por video.
    
    Parámetros:
    df (DataFrame): DataFrame que contiene las emociones y el nombre del video.
    umbral (float): Valor mínimo que una emoción debe tener para ser graficada.
    frame_rate (float): Duración en segundos de cada frame.
    """
    # Lista de columnas de emociones
    columnas_emociones = ['angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']

    # Por cada video en el DataFrame
    for video_name, group in df.groupby('Video'):
        fig, axes = plt.subplots(nrows=len(columnas_emociones), sharex=True, figsize=(15, 10))
        
        # Calcular eje x en segundos
        x_seconds = [i * frame_rate for i in range(len(group))]
        
        # Graficar cada emoción
        for idx, emocion in enumerate(columnas_emociones):
            ax = axes[idx]
            # Filtramos los datos por el umbral
            y = group[emocion].where(group[emocion] > umbral)
            ax.plot(x_seconds, y, label=emocion, color=plt.cm.tab10(idx))
            ax.set_ylim(50, 100)  # Asegurarse que el eje y empiece en 50
            ax.legend(loc='upper right')
            ax.grid(True, which='both', linestyle='--', linewidth=0.5)
            if idx == len(columnas_emociones) - 1:
                ax.set_xlabel('Tiempo (segundos)')
            ax.set_ylabel(emocion.capitalize())
            
        max_time = max(x_seconds)
        ticks_interval = 5  # Intervalo de las marcas en segundos
        x_ticks = np.arange(0, max_time + ticks_interval, ticks_interval)
        axes[-1].set_xticks(x_ticks)

        fig.suptitle(f'Progreso de Emociones en Video: {video_name}')
        plt.tight_layout()
        plt.subplots_adjust(top=0.95)
        plt.show()

In [ ]:
graficar_emociones_por_video(resultado)
graficar_sub_emociones_por_video(resultado)

In [ ]:
def obtener_porcentajes(umbral=80):
    import numpy as np
    """ 
    Calcula el porcentaje relativo con el que cada emoción supera el umbral por video.
    
    Utiliza la dataframe 'resultado' que debe existir en el entorno.
    
    Retorna:
    DataFrame: Porcentajes relativos de las emociones por video.
    """
    # Asegurarse de que 'resultado' existe en el entorno global
    if 'resultado' not in globals():
        raise ValueError("'resultado' DataFrame no se encuentra en el entorno global.")

    # Lista de columnas de emociones
    columnas_emociones = ['angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']
    
    # Seleccionamos solo las columnas de emociones y 'Video'
    datos_emociones = resultado[['Video'] + columnas_emociones].copy()
    
    # Marcar con 1 las emociones que superan el umbral y con 0 las que no
    datos_emociones[columnas_emociones] = (datos_emociones[columnas_emociones] >= umbral).astype(int)
    
    # Contar las veces que cada emoción supera el umbral
    conteo_emociones = datos_emociones.groupby('Video').sum()
    
    # Calcular el porcentaje relativo de cada emoción
    totales = conteo_emociones.sum(axis=1)
    porcentajes = (conteo_emociones.divide(totales, axis=0) * 100).round(2)
    
    # Imprimir el resumen de porcentajes para cada video
    for index, row in porcentajes.iterrows():
        print(f"Video: {index}")
        
        # Identificar y mostrar las tres emociones dominantes con sus porcentajes
        emociones_dominantes = row.nlargest(3)
        print(f"Emociones Dominantes: {', '.join([f'{emocion.capitalize()}: {value}%' for emocion, value in emociones_dominantes.items()])}")
        print("------")
        
        # Ordenar las emociones de mayor a menor porcentaje y mostrar
        for emocion, value in row.sort_values(ascending=False).items():
            print(f"{emocion.capitalize()}: {value}%")
        print("------")
    
    return porcentajes
porcentajes = obtener_porcentajes()

# Frecuencias. En conteo y en porcentaje.

In [ ]:
import pandas as pd

def obtener_estadisticas():
    """ 
    Calcula la tabla de conteos y probabilidades para las emociones por video.
    
    Utiliza la dataframe 'resultado' que debe existir en el entorno.
    
    Retorna:
    DataFrame: Tabla unificada con conteos y probabilidades para las emociones por video.
    """
    
    # Asegurarse de que 'resultado' existe en el entorno global
    if 'resultado' not in globals():
        raise ValueError("'resultado' DataFrame no se encuentra en el entorno global.")

    # Lista de columnas de emociones
    columnas_emociones = ['angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']
    
    # Definir los bins (intervalos) para las frecuencias
    bins = [0, 0.25, 0.5, 0.75, 1]
    labels = [(f"{bins[i]} - {bins[i+1]}") for i in range(len(bins)-1)]
    
    # Listas para almacenar los DataFrames de frecuencia y probabilidad de cada video
    lista_frecuencias = []
    lista_probabilidades = []

    # Para cada video, calcular las estadísticas de las emociones en los intervalos definidos
    for video, group in resultado.groupby('Video'):
        frec_video = group[columnas_emociones].apply(lambda col: pd.cut(col, bins=bins, labels=labels, right=False).value_counts(normalize=False)).fillna(0).T
        prob_video = group[columnas_emociones].apply(lambda col: pd.cut(col, bins=bins, labels=labels, right=False).value_counts(normalize=True)).fillna(0).T
        
        frec_video['Video'] = video
        prob_video['Video'] = video
        
        lista_frecuencias.append(frec_video)
        lista_probabilidades.append(prob_video)

    df_frecuencias = pd.concat(lista_frecuencias).set_index('Video', append=True).stack().unstack(level=0).unstack().fillna(0)
    df_probabilidades = pd.concat(lista_probabilidades).set_index('Video', append=True).stack().unstack(level=0).unstack().fillna(0).round(4)
    
    # Unificar ambos DataFrames
    df_frecuencias.columns = pd.MultiIndex.from_tuples([(col[0], col[1], 'Conteo') for col in df_frecuencias.columns])
    df_probabilidades.columns = pd.MultiIndex.from_tuples([(col[0], col[1], 'Porcentaje') for col in df_probabilidades.columns])
    
    df_unificado = pd.concat([df_frecuencias, df_probabilidades], axis=1)
    
    # "Aplanar" el MultiIndex y reorganizar
    df_unificado = df_unificado.stack(level=0).stack(level=0).unstack(level=-2).sort_index(axis=1)
    
    return df_unificado

# Ajustar opciones de visualización
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.max_rows', None)     # Mostrar todas las filas
pd.set_option('display.expand_frame_repr', False)  # No truncar la tabla

estadisticas = obtener_estadisticas()
estadisticas

# EXTRAS

# Funciones Incorporadas.

**Emocion_dominante:**	Identifica la emoción que prevalece durante la mayor parte del video o en segmentos específicos. Esto puede ayudar a entender la temática general del video o de partes clave del mismo.

# Proximas Funciones.

**Analisis_sentimiento_texto:**	Si el video contiene subtítulos o diálogos escritos, esta función realiza un análisis de sentimiento del texto para complementar y reforzar la detección facial de emociones.

**Integracion_APIs_redes:**	Integra APIs de plataformas de redes sociales para recopilar reacciones y comentarios sobre un video publicado y los correlaciona con las emociones detectadas en el mismo, ofreciendo un análisis completo del impacto del contenido.	

In [ ]:
emotions_dict = {
    'English': {'Angry': 'Angry', 'Disgust': 'Disgust', 'Fear': 'Fear', 'Sad': 'Sad', 'Neutral': 'Neutral', 'Surprise': 'Surprise', 'Happy': 'Happy'},
    'Spanish': {'Angry': 'Enojado', 'Disgust': 'Asco', 'Fear': 'Miedo', 'Sad': 'Triste', 'Neutral': 'Neutral', 'Surprise': 'Sorpresa', 'Happy': 'Feliz'},
    'French': {'Angry': 'En colère', 'Disgust': 'Dégoût', 'Fear': 'Peur', 'Sad': 'Triste', 'Neutral': 'Neutre', 'Surprise': 'Surprise', 'Happy': 'Heureux'},
    'Chinese (Simplified)': {'Angry': '生气', 'Disgust': '厌恶', 'Fear': '害怕', 'Sad': '伤心', 'Neutral': '中性', 'Surprise': '惊讶', 'Happy': '高兴'},
    'Arabic': {'Angry': 'غاضب', 'Disgust': 'اشمئزاز', 'Fear': 'خوف', 'Sad': 'حزين', 'Neutral': 'محايد', 'Surprise': 'مفاجأة', 'Happy': 'سعيد'},
    'Portuguese': {'Angry': 'Bravo', 'Disgust': 'Desgosto', 'Fear': 'Medo', 'Sad': 'Triste', 'Neutral': 'Neutro', 'Surprise': 'Surpresa', 'Happy': 'Feliz'},
    'Russian': {'Angry': 'Злой', 'Disgust': 'Отвращение', 'Fear': 'Страх', 'Sad': 'Грустный', 'Neutral': 'Нейтральный', 'Surprise': 'Сюрприз', 'Happy': 'Счастливый'},
    'Japanese': {'Angry': '怒っている', 'Disgust': '嫌悪', 'Fear': '恐れ', 'Sad': '悲しい', 'Neutral': 'ニュートラル', 'Surprise': '驚き', 'Happy': '幸せ'},
    'German': {'Angry': 'Wütend', 'Disgust': 'Ekel', 'Fear': 'Angst', 'Sad': 'Traurig', 'Neutral': 'Neutral', 'Surprise': 'Überraschung', 'Happy': 'Glücklich'},
    'Korean': {'Angry': '화나다', 'Disgust': '혐오', 'Fear': '두려움', 'Sad': '슬프다', 'Neutral': '중립', 'Surprise': '놀라다', 'Happy': '행복하다'},
    'Italian': {'Angry': 'Arrabbiato', 'Disgust': 'Disgusto', 'Fear': 'Paura', 'Sad': 'Triste', 'Neutral': 'Neutrale', 'Surprise': 'Sorpresa', 'Happy': 'Felice'},
    'Dutch': {'Angry': 'Boos', 'Disgust': 'Afschuw', 'Fear': 'Vrees', 'Sad': 'Treurig', 'Neutral': 'Neutraal', 'Surprise': 'Verrassing', 'Happy': 'Gelukkig'},
    'Turkish': {'Angry': 'Kızgın', 'Disgust': 'Tiksinme', 'Fear': 'Korku', 'Sad': 'Üzgün', 'Neutral': 'Notr', 'Surprise': 'Şaşırtıcı', 'Happy': 'Mutlu'},
    'Swedish': {'Angry': 'Arg', 'Disgust': 'Äcklad', 'Fear': 'Rädd', 'Sad': 'Ledsen', 'Neutral': 'Neutral', 'Surprise': 'Överraskad', 'Happy': 'Glad'},
    'Polish': {'Angry': 'Zły', 'Disgust': 'Wstręt', 'Fear': 'Strach', 'Sad': 'Smutny', 'Neutral': 'Neutralny', 'Surprise': 'Zaskoczenie', 'Happy': 'Szczęśliwy'},
    'Indonesian': {'Angry': 'Marah', 'Disgust': 'Jijik', 'Fear': 'Takut', 'Sad': 'Sedih', 'Neutral': 'Netral', 'Surprise': 'Kejutan', 'Happy': 'Bahagia'},
    'Hindi': {'Angry': 'नाराज', 'Disgust': 'घृणा', 'Fear': 'डर', 'Sad': 'दुखी', 'Neutral': 'तटस्थ', 'Surprise': 'आश्चर्य', 'Happy': 'खुश'},
    'Bengali': {'Angry': 'রাগ', 'Disgust': 'ঘৃণা', 'Fear': 'ভয়', 'Sad': 'দুঃখিত', 'Neutral': 'নিরপেক্ষ', 'Surprise': 'অবাক', 'Happy': 'খুশি'},
    'Persian': {'Angry': 'عصبانی', 'Disgust': 'نفرت', 'Fear': 'ترس', 'Sad': 'غمگین', 'Neutral': 'خنثی', 'Surprise': 'سورپرایز', 'Happy': 'خوشحال'},
    'Thai': {'Angry': 'โกรธ', 'Disgust': 'ขยะแขยง', 'Fear': 'กลัว', 'Sad': 'เศร้า', 'Neutral': 'กลาง', 'Surprise': 'ประหลาดใจ', 'Happy': 'สุขสันต์'},
    'Ukrainian': {'Angry': 'Сердитий', 'Disgust': 'Відраза', 'Fear': 'Страх', 'Sad': 'Смуток', 'Neutral': 'Нейтральний', 'Surprise': 'Здивування', 'Happy': 'Щасливий'},
    'Punjabi': {'Angry': 'ਗੁੱਸਾ', 'Disgust': 'ਘ੍ਰਿਣਾ', 'Fear': 'ਡਰ', 'Sad': 'ਦੁੱਖੀ', 'Neutral': 'ਨਿਰਪੇਖਸ਼', 'Surprise': 'ਹੈਰਾਨੀ', 'Happy': 'ਖੁਸ਼'},
    'Tamil': {'Angry': 'கோபம்', 'Disgust': 'அருவருப்பு', 'Fear': 'பயம்', 'Sad': 'சோகம்', 'Neutral': 'நியாயமான', 'Surprise': 'அதிர்ச்சி', 'Happy': 'மகிழ்ச்சி'},
    'Malayalam': {'Angry': 'കോപം', 'Disgust': 'വെറുപ്പ്', 'Fear': 'ഭയം', 'Sad': 'ദു: ഖം', 'Neutral': 'സ്വതന്ത്രം', 'Surprise': 'അത്ഭുതം', 'Happy': 'സന്തോഷം'},
    'Gujarati': {'Angry': 'ગુસ્સો', 'Disgust': 'ઘૃણા', 'Fear': 'ડર', 'Sad': 'દુ: ખિત', 'Neutral': 'સ્થિર', 'Surprise': 'આશ્ચર્ય', 'Happy': 'ખુશ'},
    'Kannada': {'Angry': 'ಕೋಪ', 'Disgust': 'ಅಸಹಾನತೆ','Fear': 'ಭಯ','Sad': 'ದುಃಖ', 'Neutral': 'ನಿಷ್ಪಕ್ಷ','Surprise': 'ಆಶ್ಚರ್ಯ','Happy': 'ಸಂತೋಷ'},
    'Oriya': {'Angry': 'କ୍ରୋଧ','Disgust': 'ଜୁଘାଣ','Fear': 'ଭୟ','Sad': 'ଦୁଃଖିତ','Neutral': 'ନିର୍ଦଳ','Surprise': 'ଆଶ୍ଚର୍ଯ୍ୟ','Happy': 'ଆନନ୍ଦ'},
    'Telugu': {'Angry': 'కోపం','Disgust': 'అసహ్యం','Fear': 'భయం','Sad': 'దుఃఖం', 'Neutral': 'నిర్వాధి','Surprise': 'ఆశ్చర్యం','Happy': 'సంతోషం'},
    'Urdu': {'Angry': 'غصہ', 'Disgust': 'نفرت', 'Fear': 'خوف', 'Sad': 'اداس', 'Neutral': 'غیر جانبدار', 'Surprise': 'حیرت', 'Happy': 'خوشی'},
}

In [ ]:
emotions_dict

No	Language	Angry	Disgust	Fear	Sad	Neutral	Surprise	Happy
1	English	Angry	Disgust	Fear	Sad	Neutral	Surprise	Happy
2	Spanish	Enojado	Asco	Miedo	Triste	Neutral	Sorpresa	Feliz
3	French	En colère	Dégoût	Peur	Triste	Neutre	Surprise	Heureux
4	Chinese (Simplified)	生气	厌恶	害怕	伤心	中性	惊讶	高兴
5	Arabic	غاضب	اشمئزاز	خوف	حزين	محايد	مفاجأة	سعيد
6	Portuguese	Bravo	Desgosto	Medo	Triste	Neutro	Surpresa	Feliz
7	Russian	Злой	Отвращение	Страх	Грустный	Нейтральный	Сюрприз	Счастливый
8	Japanese	怒っている	嫌悪	恐れ	悲しい	ニュートラル	驚き	幸せ
9	German	Wütend	Ekel	Angst	Traurig	Neutral	Überraschung	Glücklich
10	Korean	화나다	혐오	두려움	슬프다	중립	놀라다	행복하다
11	Italian	Arrabbiato	Disgusto	Paura	Triste	Neutrale	Sorpresa	Felice
12	Dutch	Boos	Afschuw	Vrees	Treurig	Neutraal	Verrassing	Gelukkig
13	Turkish	Kızgın	Tiksinme	Korku	Üzgün	Notr	Şaşırtıcı	Mutlu
14	Swedish	Arg	Äcklad	Rädd	Ledsen	Neutral	Överraskad	Glad
15	Polish	Zły	Wstręt	Strach	Smutny	Neutralny	Zaskoczenie	Szczęśliwy
16	Indonesian	Marah	Jijik	Takut	Sedih	Netral	Kejutan	Bahagia
17	Hindi	नाराज	घृणा	डर	दुखी	तटस्थ	आश्चर्य	खुश
18	Bengali	রাগ	ঘৃণা	ভয়	দু: খিত	নিরপেক্ষ	অবাক	খুশি
19	Persian	عصبانی	نفرت	ترس	غمگین	خنثی	سورپرایز	خوشحال
20	Thai	โกรธ	ขยะแขยง	กลัว	เศร้า	กลาง	ประหลาดใจ	สุขสันต์
21	Ukrainian	Сердитий	Відраза	Страх	Смуток	Нейтральний	Здивування	Щасливий
22	Bengali	রাগ	জ্বালাতন	ভয়	দুঃখিত	নিরপেক্ষ	অবাক	খুশি
23	Punjabi	ਗੁੱਸਾ	ਘ੍ਰਿਣਾ	ਡਰ	ਦੁੱਖੀ	ਨਿਰਪੇਖਸ਼	ਹੈਰਾਨੀ	ਖੁਸ਼
24	Tamil	கோபம்	அருவருப்பு	பயம்	சோகம்	நியாயமான	அதிர்ச்சி	மகிழ்ச்சி
25	Malayalam	കോപം	വെറുപ്പ്	ഭയം	ദു: ഖം	സ്വതന്ത്രം	അത്ഭുതം	സന്തോഷം
26	Gujarati	ગુસ્સો	ઘૃણા	ડર	દુ: ખિત	સ્થિર	આશ્ચર્ય	ખુશ
27	Kannada	ಕೋಪ	ಅಸಹಾನತೆ	ಭಯ	ದುಃಖ	ನಿಷ್ಪಕ್ಷ	ಆಶ್ಚರ್ಯ	ಸಂತೋಷ
28	Oriya	କ୍ରୋଧ	ଜୁଘାଣ	ଭୟ	ଦୁଃଖିତ	ନିର୍ଦଳ	ଆଶ୍ଚର୍ଯ୍ୟ	ଆନନ୍ଦ
29	Telugu	కోపం	అసహ్యం	భయం	దుఃఖం	నిర్వాధి	ఆశ్చర్యం	సంతోషం
30	Urdu	غصہ	نفرت	خوف	اداس	غیر جانبدار	حیرت	خوشی

| **Nombre de la Función** | **Descripción Detallada** | **Seleccionado** | **Puntaje Potencial** | **Justificación** |
|--------------------------|---------------------------|------------------|-----------------------|-------------------|
| `video_resumen_emocional` | Esta función analizará el video completo identificando aquellos segmentos con picos emocionales. Utilizará algoritmos avanzados de detección para seleccionar estos momentos y condensará estos segmentos en un video resumido. El resultado es un video corto que destaca los momentos con las cargas emocionales más intensas. | X | 10 | Es esencial para quienes desean ver rápidamente los puntos más emocionales sin ver el video completo. |
| `deteccion_cambio_emocional` | Se centra en identificar y señalar momentos en el video donde hay cambios bruscos o notables de emoción. Estos puntos de inflexión pueden indicar momentos de interés, sorpresa o relevancia en el contenido. | X | 9 | Para un analista de video, identificar puntos de inflexión emocional es crucial. |
| `recomendador_mejoras` | Basándose en el análisis emocional del video, sugiere modificaciones para potenciar o modificar ciertas emociones. Identifica áreas donde las emociones no se alinean con la intención del creador y sugiere cambios en ritmo, música, edición, etc. | X | 9 | Saber cómo mejorar un video basándose en las emociones es valioso para optimizar contenidos. |
| `analisis_sentimiento_texto` | Cuando un video tiene diálogos, subtítulos o texto, esta función analiza ese texto para determinar el sentimiento. Usa procesamiento de lenguaje natural para identificar el sentimiento general del texto y puede segmentar el análisis por partes del video. | X | 9 | Es una capa adicional de análisis que complementa el análisis facial, especialmente útil si hay diálogos. |
| `analisis_audio` | Analiza aspectos del audio como tono, volumen, ritmo, etc., y busca correlaciones con las emociones detectadas en el video. Esto ayuda a entender cómo el audio influye en la percepción emocional del contenido. | | 8 | Comprender la relación entre audio y emoción es fundamental en el análisis de video. |
| `comparar_videos` | Permite cargar dos o más videos y observar una comparación de la evolución emocional en cada uno. Es ideal para ver diferencias en versiones de un video o contenidos similares de diferentes creadores. | X | 8 | Los analistas a menudo comparan videos para entender tendencias o diferencias en contenidos similares. |
| `deteccion_contexto` | Analiza elementos más allá de las caras y emociones, como el fondo, acciones y escenarios del video, para proporcionar una capa adicional de análisis. Ayuda a entender cómo el contexto general del video puede influir en la percepción emocional. | | 8 | Entender el contexto puede ser clave para comprender las emociones detectadas. |
| `integracion_APIs_redes` | Se conectará a plataformas de redes sociales para recopilar datos de reacciones y comentarios relacionados con un video específico. Correlaciona estas reacciones con las emociones detectadas en el video para obtener una visión completa del impacto del contenido. | X | 8 | Las reacciones en redes pueden ofrecer insights adicionales, pero no son el foco principal en análisis puro de video. |
| `etiquetado_manual` | A través de una interfaz, los usuarios pueden etiquetar manualmente segmentos del video con emociones específicas. Esto luego se compara con la detección automática, ofreciendo una forma de validar y ajustar el análisis. | | 7 | Aunque es útil para la precisión, puede ser considerado un proceso más tedioso. |
| `emocion_por_grupo_demografico` | Si es posible identificar grupos demográficos en el video, esta función desglosará el análisis emocional mostrando cómo diferentes grupos pueden reaccionar de manera diferente al mismo contenido. | | 7 | Si bien es útil, puede no ser central para todos los análisis de video. |


| **Nombre de la Función** | **Descripción** | **Seleccionado** | **Puntaje Potencial** | **Justificación** |
|--------------------------|-----------------|------------------|-----------------------|-------------------|
| `deteccion_cambio_emocional` | Detecta y señala puntos en el video donde hay cambios bruscos de emoción. |X | 8 | Cambios emocionales bruscos suelen ser puntos clave en un video, atrayendo la atención del usuario. |
| `video_resumen_emocional` | Genera un video resumido con momentos emocionales intensos. | X | 9 | Resumir contenido es vital en la era digital donde los usuarios prefieren contenidos cortos y directos. |
| `analisis_audio` | Analiza características del audio y las correlaciona con emociones. | | 7 | El audio juega un papel crucial en la percepción emocional, aunque no todos los videos pueden beneficiarse por igual de esta función. |
| `comparar_videos` | Compara la evolución emocional en múltiples videos. | | 7 | Útil para creadores que buscan optimizar o comparar sus contenidos, pero no es central para todos. |
| `etiquetado_manual` | Interfaz para etiquetado manual de emociones en video. | | 6 | Beneficioso para una revisión detallada, pero puede ser laborioso para usuarios generales. |
| `deteccion_contexto` | Analiza el contexto del video que influye en las emociones. | | 7 | Proporciona una capa adicional de análisis, pero depende de la precisión de la detección. |
| `recomendador_mejoras` | Sugiere modificaciones al video basado en análisis emocional. |X | 9 | Ofrecer recomendaciones concretas puede ser altamente valorado por creadores de contenido. |
| `analisis_sentimiento_texto` | Análisis de sentimiento de texto en el video. | X | 8 | Complementa la detección facial, ampliando el espectro de análisis emocional. |
| `emocion_por_grupo_demografico` | Desglosa análisis emocional por grupo demográfico. | X | 8 | Ofrece insights valiosos sobre cómo distintos grupos perciben un contenido. |
| `integracion_APIs_redes` | Integra APIs de redes sociales para analizar reacciones. | X | 10 | Las redes sociales son cruciales para el marketing y la percepción de contenidos. Analizar su impacto puede ser esencial. |



1. video_resumen_emocional - 10 puntos
Descripción: Esta función analizará el video completo para identificar aquellos segmentos que tienen los picos emocionales más altos. Utilizará algoritmos avanzados de detección de emociones para seleccionar estos momentos y luego condensará estos segmentos en un video resumido. El resultado será un video corto que muestra solo los momentos con las cargas emocionales más intensas o relevantes.

Cómo funciona:

El algoritmo primero descompone el video en segmentos (por ejemplo, de unos segundos cada uno).
Analiza las emociones detectadas en cada segmento.
Identifica y selecciona aquellos segmentos que superen un cierto umbral de intensidad emocional.
Une estos segmentos para crear un video resumido.
El usuario puede tener la opción de ajustar el umbral de selección y la duración final deseada del video resumido.


In [ ]:
"""
OLD2.
# ANALIZADOR DE EMOCIONES V1.
def analizar_emociones_videos(ruta_carpeta_videos):

    ruta_temp_data = os.path.join(ruta_carpeta_videos, "temp_data")

    if not os.path.exists(ruta_temp_data):
        os.makedirs(ruta_temp_data)

    lista_archivos_videos = glob.glob(os.path.join(ruta_carpeta_videos, '*.mp4'))
    m = len(lista_archivos_videos)

    df_BD = pd.DataFrame(columns=['Nombre de Imagen', 'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy', 'Video', 'Q_images'])

    videos = [archivo for archivo in os.listdir(ruta_carpeta_videos) if archivo.endswith('.mp4')]

    for video in videos:
        ruta_video = os.path.join(ruta_carpeta_videos, video)

        def getFrame(sec):
            vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
            hasFrames, image = vidcap.read()
            if hasFrames:
                cv2.imwrite(os.path.join(ruta_temp_data, "image" + str(count) + ".jpg"), image)
            return hasFrames

        vidcap = cv2.VideoCapture(ruta_video)
        fps = vidcap.get(cv2.CAP_PROP_FPS)
        frameRate = 1 / fps 
        sec = 0
        count = 1
        success = True

        while success:
            sec = sec + frameRate
            sec = round(sec, 2)
            success = getFrame(sec)
            count = count + 1

        vidcap.release()
        cv2.destroyAllWindows()

        lista_archivos = glob.glob(os.path.join(ruta_temp_data, '*.jpg'))
        n = len(lista_archivos)

        data = []

        for i in range(1, n+1):
            img_path = os.path.join(ruta_temp_data, f"image{i}.jpg")
            
            try:
                objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=True)
                emociones = objs[0]['emotion']
            except:
                emociones = {
                    'angry': 0,
                    'disgust': 0,
                    'fear': 0,
                    'sad': 0,
                    'neutral': 0,
                    'surprise': 0,
                    'happy': 0
                }
            
            #objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)

            nombre_imagen = f"imagen{i}.jpg"
            #emociones = objs[0]['emotion']
            data.append([nombre_imagen, emociones])

        df = pd.DataFrame(data, columns=['Nombre de Imagen', 'Emociones'])
        df = df.join(df['Emociones'].apply(pd.Series))
        df = df.drop('Emociones', axis=1)
        df = df.round(1)
        nuevo_orden = ['Nombre de Imagen', 'happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear']
        df = df[nuevo_orden]
        df = df.assign(Video=video.split('.')[0])
        df = df.assign(Q_images=n)
    
        df_BD = pd.concat([df_BD, df], ignore_index=True)

        archivos = os.listdir(ruta_temp_data)
        for archivo in archivos:
            if archivo.endswith('.jpg'):
                ruta_archivo = os.path.join(ruta_temp_data, archivo)
                os.remove(ruta_archivo)

    return df_BD


def insertar_emocion(image, emociones, umbral=80):
    """
    Inserta la emoción detectada en la imagen si supera el umbral.
    """
    for emocion, valor in emociones.items():
        if valor >= umbral:
            # Ajustar la posición del texto
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 1.0  # Duplicamos el tamaño
            thickness = 4  # Ajustar el grosor del texto acorde al tamaño
            color = (0, 255, 255)  # Amarillo
            border_color = (0, 0, 0)  # Negro para el borde
            
            # Cambiar la primera letra a mayúscula y agregar el porcentaje
            texto = f"{emocion.capitalize()} - {valor:.0f}%"
            text_width, text_height = cv2.getTextSize(texto, font, scale, thickness)[0]
            position = (int((image.shape[1] - text_width) / 2), image.shape[0] - 50)  # Centrado horizontal y abajo
            
            # Dibujar el borde del texto
            cv2.putText(image, texto, position, font, scale, border_color, thickness+2, cv2.LINE_AA)
            # Dibujar el texto
            cv2.putText(image, texto, position, font, scale, color, thickness, cv2.LINE_AA)
            break
    return image



def generar_video_con_subtitulos(df, ruta_carpeta_videos, umbral=80):
    """
    Genera un video con subtítulos basado en un DataFrame que contiene análisis de emociones.
    """

    for video_name in df['Video'].unique():
        ruta_video = os.path.join(ruta_carpeta_videos, video_name + ".mp4")
        vidcap = cv2.VideoCapture(ruta_video)
        fps = vidcap.get(cv2.CAP_PROP_FPS)

        # Iniciar video de salida
        fourcc = cv2.VideoWriter_fourcc(*'MP4V')
        output_video_path = os.path.join(ruta_carpeta_videos, f"processed_{video_name}.mp4")
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (int(vidcap.get(3)), int(vidcap.get(4))))

        # Filtrar el DataFrame para el video actual
        video_data = df[df['Video'] == video_name]
        
        for index, row in video_data.iterrows():
            success, image = vidcap.read()
            if success:
                emociones = {
                    'angry': row['angry'],
                    'disgust': row['disgust'],
                    'fear': row['fear'],
                    'sad': row['sad'],
                    'neutral': row['neutral'],
                    'surprise': row['surprise'],
                    'happy': row['happy']
                }
                imagen_con_subtitulo = insertar_emocion(image, emociones, umbral)
                out.write(imagen_con_subtitulo)
            else:
                break

        vidcap.release()
        out.release()
        
        

# Para usar la función:
resultado = analizar_emociones_videos("C:/Users/franc/Desktop/DeepVideoFER/Gestion")

# Uso de las funciones:
generar_video_con_subtitulos(resultado, "C:/Users/franc/Desktop/DeepVideoFER/Gestion")

print(resultado)
print("Se han creado los videos con emosubtítulos.")

"""

In [ ]:
"""
    'Polish': {'Angry': 'Zły', 'Disgust': 'Wstręt', 'Fear': 'Strach', 'Sad': 'Smutny', 'Neutral': 'Neutralny', 'Surprise': 'Zaskoczenie', 'Happy': 'Szczęśliwy'},
    'Korean': {'Angry': '화나다', 'Disgust': '혐오', 'Fear': '두려움', 'Sad': '슬프다', 'Neutral': '중립', 'Surprise': '놀라다', 'Happy': '행복하다'},
    'Russian': {'Angry': 'Злой', 'Disgust': 'Отвращение', 'Fear': 'Страх', 'Sad': 'Грустный', 'Neutral': 'Нейтральный', 'Surprise': 'Сюрприз', 'Happy': 'Счастливый'},
    'Japanese': {'Angry': '怒っている', 'Disgust': '嫌悪', 'Fear': '恐れ', 'Sad': '悲しい', 'Neutral': 'ニュートラル', 'Surprise': '驚き', 'Happy': '幸せ'},
    'Arabic': {'Angry': 'غاضب', 'Disgust': 'اشمئزاز', 'Fear': 'خوف', 'Sad': 'حزين', 'Neutral': 'محايد', 'Surprise': 'مفاجأة', 'Happy': 'سعيد'},
    'Hindi': {'Angry': 'नाराज', 'Disgust': 'घृणा', 'Fear': 'डर', 'Sad': 'दुखी', 'Neutral': 'तटस्थ', 'Surprise': 'आश्चर्य', 'Happy': 'खुश'},
    'Bengali': {'Angry': 'রাগ', 'Disgust': 'ঘৃণা', 'Fear': 'ভয়', 'Sad': 'দুঃখিত', 'Neutral': 'নিরপেক্ষ', 'Surprise': 'অবাক', 'Happy': 'খুশি'},
    'Persian': {'Angry': 'عصبانی', 'Disgust': 'نفرت', 'Fear': 'ترس', 'Sad': 'غمگین', 'Neutral': 'خنثی', 'Surprise': 'سورپرایز', 'Happy': 'خوشحال'},
    'Thai': {'Angry': 'โกรธ', 'Disgust': 'ขยะแขยง', 'Fear': 'กลัว', 'Sad': 'เศร้า', 'Neutral': 'กลาง', 'Surprise': 'ประหลาดใจ', 'Happy': 'สุขสันต์'},
    'Ukrainian': {'Angry': 'Сердитий', 'Disgust': 'Відраза', 'Fear': 'Страх', 'Sad': 'Смуток', 'Neutral': 'Нейтральний', 'Surprise': 'Здивування', 'Happy': 'Щасливий'},
    'Punjabi': {'Angry': 'ਗੁੱਸਾ', 'Disgust': 'ਘ੍ਰਿਣਾ', 'Fear': 'ਡਰ', 'Sad': 'ਦੁੱਖੀ', 'Neutral': 'ਨਿਰਪੇਖਸ਼', 'Surprise': 'ਹੈਰਾਨੀ', 'Happy': 'ਖੁਸ਼'},
    'Tamil': {'Angry': 'கோபம்', 'Disgust': 'அருவருப்பு', 'Fear': 'பயம்', 'Sad': 'சோகம்', 'Neutral': 'நியாயமான', 'Surprise': 'அதிர்ச்சி', 'Happy': 'மகிழ்ச்சி'},
    'Malayalam': {'Angry': 'കോപം', 'Disgust': 'വെറുപ്പ്', 'Fear': 'ഭയം', 'Sad': 'ദു: ഖം', 'Neutral': 'സ്വതന്ത്രം', 'Surprise': 'അത്ഭുതം', 'Happy': 'സന്തോഷം'},
    'Gujarati': {'Angry': 'ગુસ્સો', 'Disgust': 'ઘૃણા', 'Fear': 'ડર', 'Sad': 'દુ: ખિત', 'Neutral': 'સ્થિર', 'Surprise': 'આશ્ચર્ય', 'Happy': 'ખુશ'},
    'Kannada': {'Angry': 'ಕೋಪ', 'Disgust': 'ಅಸಹಾನತೆ','Fear': 'ಭಯ','Sad': 'ದುಃಖ', 'Neutral': 'ನಿಷ್ಪಕ್ಷ','Surprise': 'ಆಶ್ಚರ್ಯ','Happy': 'ಸಂತೋಷ'},
    'Oriya': {'Angry': 'କ୍ରୋଧ','Disgust': 'ଜୁଘାଣ','Fear': 'ଭୟ','Sad': 'ଦୁଃଖିତ','Neutral': 'ନିର୍ଦଳ','Surprise': 'ଆଶ୍ଚର୍ଯ୍ୟ','Happy': 'ଆନନ୍ଦ'},
    'Telugu': {'Angry': 'కోపం','Disgust': 'అసహ్యం','Fear': 'భయం','Sad': 'దుఃఖం', 'Neutral': 'నిర్వాధి','Surprise': 'ఆశ్చర్యం','Happy': 'సంతోషం'},
    'Urdu': {'Angry': 'غصہ', 'Disgust': 'نفرت', 'Fear': 'خوف', 'Sad': 'اداس', 'Neutral': 'غیر جانبدار', 'Surprise': 'حیرت', 'Happy': 'خوشی'},
"""